# 02a: Explore canonical telemetry

Read-only, DuckDB-backed exploration. 

Queries are filtered and capped before data reaches pandas.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'src' / 'dnsp_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the dnsp_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from dnsp_analysis.config import load_config
from dnsp_analysis.db import canonical_output_path, duplicate_audit_path
from dnsp_analysis.metadata import metadata_output_path
from dnsp_analysis.schemas import sql_string

pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')

## Select a bounded slice

Edit these values freely. `SERIAL=None` chooses a site from the selected month. `MAX_ROWS` is a hard guardrail.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'analysis.toml'
MONTH = '2024-09'
SERIAL = None
DATE_FROM = None  # local date, e.g. '2025-04-10'
DATE_TO = None    # local date, exclusive
PHASES = ['A', 'B', 'C']
MAX_ROWS = 100_000

config = load_config(CONFIG_PATH, check_inputs=False)
full_scope = config.scope(None, None)
canonical_dir = canonical_output_path(config, full_scope)
assert canonical_dir.is_dir(), f'Run notebook 01 first: {canonical_dir}'
canonical_glob = str(canonical_dir / '**' / '*.parquet')
con = duckdb.connect()
con.execute(f"SET memory_limit='{config.processing.memory_limit}'")
con.execute(f"SET threads={min(config.processing.threads, 4)}")
con.execute("SET TimeZone='UTC'")
con.execute(f"CREATE VIEW canonical AS SELECT * FROM read_parquet({sql_string(canonical_glob)}, hive_partitioning=true)")
print('Canonical source:', canonical_dir)

In [ ]:
year, month = map(int, MONTH.split('-'))
overview = con.execute('''
    SELECT count(*) n_rows, count(DISTINCT serial) n_sites,
           min(timestamp_local) first_local, max(timestamp_local) last_local,
           count_if(NOT voltage_physical_ok) invalid_voltage_rows,
           count_if(p_export_w IS NULL) null_p_rows,
           count_if(q_absorbing_var IS NULL) null_q_rows,
           count_if(NOT metadata_available) rows_without_metadata
    FROM canonical WHERE year_utc=? AND month_utc=?
''', [year, month]).fetchdf()
display(overview)

site_choices = con.execute('''
    SELECT serial, analysis_cohort, install_phase_count, count(*) n_rows,
           count_if(p_export_w IS NOT NULL) n_power_rows
    FROM canonical WHERE year_utc=? AND month_utc=?
    GROUP BY ALL ORDER BY n_power_rows DESC LIMIT 50
''', [year, month]).fetchdf()
display(site_choices.head(20))
if SERIAL is None:
    SERIAL = str(site_choices.iloc[0]['serial'])
print('Selected serial:', SERIAL)

In [ ]:
# Change SERIAL if wanted:
SERIAL = 810140828
SERIAL = 810492661

where = ['year_utc=?', 'month_utc=?', 'cast(serial as varchar)=?']
params = [year, month, str(SERIAL)]
if DATE_FROM:
    where.append('timestamp_local >= cast(? as timestamp)'); params.append(DATE_FROM)
if DATE_TO:
    where.append('timestamp_local < cast(? as timestamp)'); params.append(DATE_TO)
where.append("phase IN (SELECT unnest(?))"); params.append(PHASES)
query = f'''SELECT * FROM canonical WHERE {' AND '.join(where)} ORDER BY timestamp_utc, phase LIMIT ?'''
sample = con.execute(query, [*params, MAX_ROWS]).fetchdf()
assert len(sample) < MAX_ROWS, 'Row cap reached: narrow dates/phases before interpreting plots.'
display(sample.head())
display(sample.groupby('phase')[['voltage_v','p_export_w','q_absorbing_var','current_a']].agg(['count','median','min','max']))

In [ ]:
sample

## Time series and voltage relationships

These are net-meter P/Q against revenue-meter voltage. They are exploratory.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for phase, g in sample.groupby('phase'):
    axes[0].plot(g.timestamp_local, g.voltage_v.where(g.voltage_physical_ok), '.', ms=1, label=phase)
    axes[1].plot(g.timestamp_local, g.p_export_w, '.', ms=1, label=phase)
    axes[2].plot(g.timestamp_local, g.q_absorbing_var, '.', ms=1, label=phase)
axes[0].set_ylabel('Revenue-meter V'); axes[1].set_ylabel('Net export W'); axes[2].set_ylabel('Net absorbing VAr')
axes[0].legend(ncol=3); plt.show()

valid = sample[sample.voltage_physical_ok & sample.p_export_w.notna() & sample.q_absorbing_var.notna()]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for phase, g in valid.groupby('phase'):
    axes[0].scatter(g.voltage_v, g.p_export_w, s=2, alpha=.15, label=phase)
    axes[1].scatter(g.voltage_v, g.q_absorbing_var, s=2, alpha=.15, label=phase)
axes[0].set(xlabel='Revenue-meter V', ylabel='Net export W'); axes[1].set(xlabel='Revenue-meter V', ylabel='Net absorbing VAr')
axes[0].legend(); plt.show()

In [ ]:
daily = (sample.assign(local_hour=sample.timestamp_local.dt.hour + sample.timestamp_local.dt.minute/60)
         .groupby(['phase','local_hour'])[['voltage_v','p_export_w','q_absorbing_var']].median().reset_index())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for phase, g in daily.groupby('phase'):
    for ax, col in zip(axes, ['voltage_v','p_export_w','q_absorbing_var']): ax.plot(g.local_hour, g[col], label=phase)
axes[0].set_ylabel('Median revenue-meter V'); axes[1].set_ylabel('Median net export W'); axes[2].set_ylabel('Median net absorbing VAr')
for ax in axes: ax.set_xlabel('Local hour')
axes[0].legend(); plt.show()

quality = sample.groupby('phase').agg(
    rows=('serial','size'), p_null=('p_export_w',lambda x:x.isna().sum()),
    q_null=('q_absorbing_var',lambda x:x.isna().sum()),
    current_null=('current_a',lambda x:x.isna().sum()),
    voltage_zero=('voltage_v',lambda x:(x<=0).sum()))
display(quality)

## Sign, metadata, duplicate provenance and timezone checks

In [ ]:
sample['local_period'] = pd.cut(sample.timestamp_local.dt.hour, [-1,5,9,15,19,24], labels=['night','morning','midday','evening','late'])
display(sample.groupby(['local_period','phase'], observed=True)[['active_power_raw_w','p_export_w','reactive_power_raw_var','q_absorbing_var']].median())
print('Working signs only — active export sign:', config.assumptions.active_export_sign,
      '; reactive absorbing sign:', config.assumptions.reactive_absorbing_sign)

metadata = con.execute(f'''SELECT * FROM read_parquet({sql_string(metadata_output_path(config))}) WHERE cast(serial as varchar)=?''', [str(SERIAL)]).fetchdf()
display(metadata)
audit = duplicate_audit_path(config, full_scope)
if audit.is_file():
    display(con.execute(f'''SELECT * FROM read_parquet({sql_string(audit)}) WHERE cast(serial as varchar)=? ORDER BY row_count DESC LIMIT 25''', [str(SERIAL)]).fetchdf())

display(con.execute('''SELECT timestamp_utc, timestamp_local,
    date_diff('minute', cast(timestamp_utc AS TIMESTAMP), timestamp_local) utc_offset_minutes
    FROM canonical WHERE cast(serial as varchar)=? ORDER BY timestamp_utc LIMIT 20''', [str(SERIAL)]).fetchdf())
print('Close connection when finished: con.close()')

# Scratch book

In [ ]:
sample

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'analysis.toml'
MONTH = '2024-09'
SERIAL = None
DATE_FROM = None  # local date, e.g. '2025-04-10'
DATE_TO = None    # local date, exclusive
PHASES = ['A', 'B', 'C']
MAX_ROWS = 100_000

config = load_config(CONFIG_PATH, check_inputs=False)
full_scope = config.scope(None, None)
canonical_dir = canonical_output_path(config, full_scope)
assert canonical_dir.is_dir(), f'Run notebook 01 first: {canonical_dir}'
canonical_glob = str(canonical_dir / '**' / '*.parquet')
con = duckdb.connect()
con.execute(f"SET memory_limit='{config.processing.memory_limit}'")
con.execute(f"SET threads={min(config.processing.threads, 4)}")
con.execute("SET TimeZone='UTC'")
con.execute(f"CREATE VIEW canonical AS SELECT * FROM read_parquet({sql_string(canonical_glob)}, hive_partitioning=true)")
print('Canonical source:', canonical_dir)

In [ ]:
voltage_high_sites = con.execute("""
    SELECT CAST(serial AS VARCHAR) AS serial,
           COUNT(*) AS n_rows,
           MIN(timestamp_local) AS first_seen,
           MAX(timestamp_local) AS last_seen
    FROM canonical
    WHERE year_utc = ? AND month_utc = ?
      AND voltage_v > 253
    GROUP BY serial
    ORDER BY n_rows DESC
""", [year, month]).fetchdf()

display(voltage_high_sites)

In [ ]:
year, month = map(int, MONTH.split('-'))
overview = con.execute('''
    SELECT count(*) n_rows, count(DISTINCT serial) n_sites,
           min(timestamp_local) first_local, max(timestamp_local) last_local,
           count_if(NOT voltage_physical_ok) invalid_voltage_rows,
           count_if(p_export_w IS NULL) null_p_rows,
           count_if(q_absorbing_var IS NULL) null_q_rows,
           count_if(NOT metadata_available) rows_without_metadata
    FROM canonical WHERE year_utc=? AND month_utc=?
''', [year, month]).fetchdf()
display(overview)

site_choices = con.execute('''
    SELECT serial, analysis_cohort, install_phase_count, count(*) n_rows,
           count_if(p_export_w IS NOT NULL) n_power_rows
    FROM canonical WHERE year_utc=? AND month_utc=?
    GROUP BY ALL ORDER BY n_power_rows DESC LIMIT 50
''', [year, month]).fetchdf()
display(site_choices.head(20))
if SERIAL is None:
    SERIAL = str(site_choices.iloc[0]['serial'])
print('Selected serial:', SERIAL)